In [1]:
import pandas as pd
import numpy as np

In [2]:
df = pd.read_csv('../data/data.csv')

df.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 4800 entries, 0 to 4799
Data columns (total 64 columns):
 #   Column  Non-Null Count  Dtype  
---  ------  --------------  -----  
 0   label   4800 non-null   object 
 1   x0      4800 non-null   float64
 2   y0      4800 non-null   float64
 3   z0      4800 non-null   float64
 4   x1      4800 non-null   float64
 5   y1      4800 non-null   float64
 6   z1      4800 non-null   float64
 7   x2      4800 non-null   float64
 8   y2      4800 non-null   float64
 9   z2      4800 non-null   float64
 10  x3      4800 non-null   float64
 11  y3      4800 non-null   float64
 12  z3      4800 non-null   float64
 13  x4      4800 non-null   float64
 14  y4      4800 non-null   float64
 15  z4      4800 non-null   float64
 16  x5      4800 non-null   float64
 17  y5      4800 non-null   float64
 18  z5      4800 non-null   float64
 19  x6      4800 non-null   float64
 20  y6      4800 non-null   float64
 21  z6      4800 non-null   float64
 22  

In [3]:
df.head()

,label,x0,y0,z0,x1,y1,z1,x2,y2,z2,...,z17,x18,y18,z18,x19,y19,z19,x20,y20,z20
0,A,0.782372,0.584955,-3.838640e-07,0.724044,0.569743,-0.025130,0.678607,0.498266,-0.033284,...,0.005920,0.785460,0.318381,-0.007713,0.780306,0.353567,-0.000972,0.780108,0.384850,0.010352
1,A,0.785183,0.585897,-3.763872e-07,0.724960,0.565913,-0.024602,0.678944,0.497008,-0.032798,...,0.010546,0.784226,0.319938,-0.001743,0.779083,0.355084,0.005894,0.779805,0.385728,0.017709
2,A,0.782411,0.576275,-3.756243e-07,0.724783,0.561614,-0.026819,0.677612,0.490260,-0.035304,...,0.008733,0.785813,0.316370,-0.004468,0.779497,0.350856,0.002275,0.779441,0.379876,0.013414
3,A,0.782492,0.582599,-3.982607e-07,0.723063,0.565114,-0.024564,0.677561,0.492995,-0.033106,...,0.006132,0.784894,0.312351,-0.007883,0.783297,0.340562,-0.003004,0.782615,0.372408,0.007178
4,A,0.779827,0.580211,-3.956992e-07,0.721736,0.565600,-0.025302,0.676861,0.492315,-0.033864,...,0.004916,0.784497,0.311871,-0.008212,0.781524,0.344430,-0.001502,0.780771,0.374784,0.009794


In [4]:
X = df.drop('label', axis=1)
y = df['label']

print(X.shape)

(4800, 63)


In [5]:
def normalize_landmarks(row):
    coords = row.values.reshape(21,3)
    
    coords = coords - coords[0]
    
    max_val = np.max(np.abs(coords))
    if max_val != 0:
        coords = coords / max_val
        
    return coords.flatten()

In [6]:
X_norm = np.array([normalize_landmarks(row) for _, row in X.iterrows()])

print(X_norm.shape)

(4800, 63)


In [7]:
def compute_distances(coords):
    coords = coords.reshape(21, 3)

    def dist(i, j):
        return np.linalg.norm(coords[i] - coords[j])

    features = []

    pairs = [
        (4,8), (8,12), (12,16), (16,20),  # giữa các ngón
        (0,8), (0,12), (0,16), (0,20)     # cổ tay đến đầu ngón
    ]

    for i, j in pairs:
        features.append(dist(i, j))

    return np.array(features)

In [8]:
X_features = []

for row in X_norm:
    distances = compute_distances(row)
    combined = np.concatenate([row, distances])
    X_features.append(combined)
    
X_features = np.array(X_features)

print(X_features.shape)

(4800, 71)


In [9]:
from sklearn.model_selection import train_test_split

X_train, X_test, y_train, y_test = train_test_split(X_features,
                                                    y,
                                                    test_size=0.2,
                                                    shuffle=True,
                                                    random_state=42)

print("Số mẫu huấn luyện: ", len(X_train))
print("Số mẫu kiểm thử: ", len(X_test))

Số mẫu huấn luyện:  3840
Số mẫu kiểm thử:  960


In [ ]:
from sklearn.preprocessing import StandardScaler

scaler = StandardScaler()

X_train_scaled = scaler.fit_transform(X_train)
X_test_scaled = scaler.transform(X_test)